# 5. Risk Assessment (Multiple Return Periods)

This notebook calculates Expected Annual Damage (EAD) by combining damages from multiple flood return periods.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from damagescanner import DamageScanner

## Define input data

We use multiple hazard maps with different return periods (probability of occurrence).

In [ ]:
data_path = Path("..") / "data" / "kampen"

# Multiple hazard maps with return periods
hazard_dict = {
    10: data_path / "hazard" / "1in10_inundation_map.tif",
    50: data_path / "hazard" / "1in50_inundation_map.tif",
    100: data_path / "hazard" / "1in100_inundation_map.tif",
    500: data_path / "hazard" / "1in500_inundation_map.tif",
    1000: data_path / "hazard" / "1in1000_inundation_map.tif",
}

exposure = data_path / "exposure" / "landuse_map.tif"
curves = data_path / "vulnerability" / "curves.csv"
maxdam = data_path / "vulnerability" / "maxdam.csv"

## Initialize DamageScanner

We use one of the hazard maps for initialization (the return period doesn't matter here).

In [ ]:
ds = DamageScanner(
    hazard_data=hazard_dict[100],  # Use 1:100 for initialization
    feature_data=exposure,
    curves=curves,
    maxdam=maxdam,
)

print(f"Assessment type: {ds.assessment_type}")

## Calculate damages for each return period

In [ ]:
damages_per_rp = {}

for rp, hazard_path in hazard_dict.items():
    ds_rp = DamageScanner(
        hazard_data=hazard_path,
        feature_data=exposure,
        curves=curves,
        maxdam=maxdam,
    )
    damage_df, _, _, _ = ds_rp.calculate()
    total_damage = damage_df["damage"].sum()
    damages_per_rp[rp] = total_damage
    print(f"Return period 1:{rp}: €{total_damage:,.0f}")

## Calculate Expected Annual Damage (EAD)

Using the `risk()` method which integrates the exceedance probability curve.

In [ ]:
risk_result = ds.risk(hazard_dict)

if risk_result is not None:
    ead = risk_result["risk"].sum()
    print(f"Expected Annual Damage (EAD): €{ead:,.0f}")
else:
    print("Risk calculation returned no results")

## Visualize damage-frequency curve

In [ ]:
# Create DataFrame for plotting
df = pd.DataFrame(
    {
        "return_period": list(damages_per_rp.keys()),
        "damage": list(damages_per_rp.values()),
    }
)
df["probability"] = 1 / df["return_period"]  # Annual exceedance probability
df = df.sort_values("return_period")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Damage vs Return Period
axes[0].plot(df["return_period"], df["damage"] / 1e6, "o-", markersize=8)
axes[0].set_xlabel("Return Period (years)")
axes[0].set_ylabel("Damage (€ million)")
axes[0].set_title("Damage vs Return Period")
axes[0].set_xscale("log")
axes[0].grid(True, alpha=0.3)

# Exceedance Probability Curve
axes[1].fill_between(df["probability"], df["damage"] / 1e6, alpha=0.3)
axes[1].plot(df["probability"], df["damage"] / 1e6, "o-", markersize=8)
axes[1].set_xlabel("Annual Exceedance Probability")
axes[1].set_ylabel("Damage (€ million)")
axes[1].set_title("Exceedance Probability Curve\n(Area = EAD)")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Summary

In [ ]:
print("Flood Risk Summary")
print("=" * 40)
print("Damage per return period:")
for rp, damage in sorted(damages_per_rp.items()):
    print(f"  1:{rp:4d} year flood: €{damage:>12,.0f}")

if risk_result is not None:
    print(f"\nExpected Annual Damage: €{ead:,.0f}")